# 04 — Bronze: streets_chunk_3.json via Auto Loader

## Widgets & Configuration

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("bronze_schema", "bronze", "3. Bronze Schema")
dbutils.widgets.text("chunks_volume", "chunks", "4. Chunks Volume")

CATALOG = dbutils.widgets.get("catalog_name")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
CHUNKS_VOL = dbutils.widgets.get("chunks_volume")

TARGET_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.streets_autoloader"
CHUNKS_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{CHUNKS_VOL}"
SOURCE_FILE = f"{CHUNKS_PATH}/streets_chunk_3.json"
ISOLATED_SRC = f"{CHUNKS_PATH}/isolated_json_source"
ISOLATED_FILE = f"{ISOLATED_SRC}/streets_chunk_3.json"
CHECKPOINT_BASE = f"{CHUNKS_PATH}/streaming_metadata/streets_json"
SCHEMA_LOC = f"{CHECKPOINT_BASE}/schema"
CHECKPOINT_LOC = f"{CHECKPOINT_BASE}/chkpt"

print(f"Target table : {TARGET_TABLE}")
print(f"Source file  : {SOURCE_FILE}")
print(f"Checkpoint   : {CHECKPOINT_LOC}")

## Bronze Schema DDL
all STRING — inferSchema = false

In [0]:
BRONZE_SCHEMA_DDL = """
    noise     STRING,
    pollution STRING,
    date      STRING,
    light     STRING,
    raining   STRING,
    street_id STRING
"""
EXPECTED_KEYS = ["noise", "pollution", "date", "light", "raining", "street_id"]

##Pre-Flight Checks

In [0]:
try:
    dbutils.fs.ls(SOURCE_FILE)
except Exception:
    raise FileNotFoundError(
        f"Source file not found: {SOURCE_FILE}\n"
        f"Run 02_data_chunking first to generate the JSON chunk."
    )
print(f"Source file confirmed: {SOURCE_FILE}")

dbutils.fs.mkdirs(ISOLATED_SRC)
dbutils.fs.mkdirs(SCHEMA_LOC)
dbutils.fs.mkdirs(CHECKPOINT_LOC)
dbutils.fs.cp(SOURCE_FILE, ISOLATED_FILE, recurse=False)  # always fresh copy
print(f"File staged at: {ISOLATED_FILE}")

# Peek at actual JSON keys before committing to a schema — this is a chunk
# WE wrote in 02_data_chunking, so the keys are expected to match exactly,
# but confirming here catches a broken/re-run chunking job before Auto
# Loader silently reads garbage.
df_raw_check = spark.read.option("multiLine", "true").json(ISOLATED_FILE)
actual_keys = df_raw_check.columns
print(f"Actual JSON keys: {actual_keys}")

if set(actual_keys) != set(EXPECTED_KEYS):
    raise ValueError(
        f"JSON keys do not match expectation.\n"
        f"Expected: {EXPECTED_KEYS}\nActual:   {actual_keys}\n"
        f"Re-run 02_data_chunking — do not proceed with a schema guess."
    )

preflight_count = df_raw_check.filter(
    F.col("street_id").isNotNull() & F.col("date").isNotNull()
).count()
if preflight_count == 0:
    raise ValueError("JSON has 0 valid rows (street_id/date both present) — check the chunk file.")
print(f"Pre-flight PASSED — {preflight_count:,} valid rows, ready to stream.")

# Reset checkpoint every run — reprocessing is safe because the merge below
# is idempotent on (street_id, date), not because the checkpoint skips files.
dbutils.fs.rm(CHECKPOINT_BASE, recurse=True)
dbutils.fs.mkdirs(SCHEMA_LOC)
dbutils.fs.mkdirs(CHECKPOINT_LOC)
print(f"Checkpoint reset: {CHECKPOINT_BASE}")

## Idempotent Micro-Batch Function

In [0]:
# First run: table doesn't exist -> created from the first batch.
# Re-run: table exists -> MERGE on (street_id, date) — matching pairs are
# left alone (source data doesn't change), new pairs are inserted. Safe to
# run this notebook as many times as you want against the same chunk file.

def upsert_to_bronze(micro_batch_df, batch_id):
    if "_rescued_data" in micro_batch_df.columns:
        micro_batch_df = micro_batch_df.drop("_rescued_data")

    micro_batch_df = micro_batch_df.filter(
        F.col("street_id").isNotNull() & F.col("date").isNotNull()
    )
    batch_count = micro_batch_df.count()
    print(f"  Batch {batch_id} — {batch_count:,} valid rows after null guard")

    if batch_count == 0:
        print(f"  Batch {batch_id} — empty after null guard, skipping.")
        return

    if not spark.catalog.tableExists(TARGET_TABLE):
        print(f"  First run — creating table: {TARGET_TABLE}")
        (micro_batch_df.write
            .format("delta").mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TARGET_TABLE))
        print(f"  Table created — {batch_count:,} rows written.")
    else:
        print(f"  Table exists — running MERGE on (street_id, date)...")
        target = DeltaTable.forName(spark, TARGET_TABLE)
        (target.alias("t")
            .merge(micro_batch_df.alias("s"),
                   "t.street_id = s.street_id AND t.date = s.date")
            .whenNotMatchedInsertAll()
            .execute())
        print(f"  MERGE complete — new (street_id, date) pairs inserted, existing pairs skipped.")


## Auto Loader Read Stream

In [0]:
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOC)
    .option("cloudFiles.inferColumnTypes", "false")
    .option("multiLine", "true")
    .schema(BRONZE_SCHEMA_DDL)
    .load(ISOLATED_SRC)
    .withColumn("load_dt", F.current_timestamp())
    .withColumn("source", F.col("_metadata.file_name")))

print(f"Starting stream -> {TARGET_TABLE} ...")
query = (df_stream.writeStream
    .foreachBatch(upsert_to_bronze)
    .option("checkpointLocation", CHECKPOINT_LOC)
    .trigger(availableNow=True)
    .start())
query.awaitTermination()
print("Stream complete.")

## Verification & Audit

In [0]:
if not spark.catalog.tableExists(TARGET_TABLE):
    raise Exception(
        f"Table still not found after stream: {TARGET_TABLE}\n"
        f"Check for a 'Batch 0' line above — if none printed, the isolated "
        f"source folder was empty."
    )

df_bronze = spark.table(TARGET_TABLE)
total = df_bronze.count()
null_street_id = df_bronze.filter(F.col("street_id").isNull()).count()
null_date = df_bronze.filter(F.col("date").isNull()).count()
no_load_dt = df_bronze.filter(F.col("load_dt").isNull()).count()
no_source = df_bronze.filter(F.col("source").isNull()).count()
dupe_grain = total - df_bronze.select("street_id", "date").distinct().count()

print(f"\n{'='*60}\n  INGESTION SUMMARY\n{'='*60}")
print(f"  Table               : {TARGET_TABLE}")
print(f"  Total rows          : {total:,}")
print(f"{'='*60}\n  NULL / GRAIN CHECKS")
print(f"  null street_id      : {null_street_id:,}   (must be 0)")
print(f"  null date           : {null_date:,}   (must be 0)")
print(f"  duplicate grain rows: {dupe_grain:,}   (must be 0)")
print(f"{'='*60}\n  AUDIT COLUMNS")
print(f"  missing load_dt     : {no_load_dt:,}   (must be 0)")
print(f"  missing source      : {no_source:,}   (must be 0)")
print(f"{'='*60}")
print(f"  Idempotency         : MERGE on (street_id, date)")
print(f"{'='*60}")

all_pass = (null_street_id == 0 and null_date == 0 and dupe_grain == 0
            and no_load_dt == 0 and no_source == 0 and total > 0)
print("  ALL CHECKS PASSED" if all_pass else "  WARNING: one or more checks failed — see above.")

display(spark.table(TARGET_TABLE).limit(10))